# Pipeline walkthrough

This notebook runs the pipeline stage by stage and shows the results. Reviewers read this first.

Rules: reuse the code the DAG uses (import it), show evidence after each stage, keep the outputs when you commit.

Replace every *TODO* below. Add cells freely.

In [1]:
import os, sys
import pandas as pd

sys.path.insert(0, "/opt/airflow")  # so `ingestion` and `dags` import the same way Airflow sees them

import psycopg2

def query(sql, params=None):
    with psycopg2.connect(
        host=os.environ["WAREHOUSE_HOST"], port=os.environ["WAREHOUSE_PORT"],
        dbname=os.environ["WAREHOUSE_DB"], user=os.environ["WAREHOUSE_USER"], password=os.environ["WAREHOUSE_PASSWORD"],
    ) as conn, conn.cursor() as cur:
        cur.execute(sql, params)
        return cur.fetchall() if cur.description else None

query("select version()")

[('PostgreSQL 16.15 on x86_64-pc-linux-musl, compiled by gcc (Alpine 15.2.0) 15.2.0, 64-bit',)]

In [2]:
from datetime import date

start_date = date(2026, 8, 25)
end_date = date(2026, 9, 23)
logical_date = end_date.isoformat()

print("Single-date walkthrough:", logical_date)
print("30-day backfill:", start_date, "to", end_date)

Single-date walkthrough: 2026-09-23
30-day backfill: 2026-08-25 to 2026-09-23


## 1. Extract

This stage demonstrates the API extraction logic used by the Airflow DAG.

- **Input:** one logical date.
- **Cities:** all cities defined in `config/cities.yml`.
- **Source:** Open-Meteo Archive API.
- **Granularity:** one API response per city for the requested logical date.
- **Concurrency:** requests for the five cities are executed concurrently because they are independent, I/O-bound HTTP calls.
- **Timeouts:** each HTTP request uses a 5-second connection timeout and a 30-second read timeout.
- **Retries:** transient HTTP failures and rate limiting (`429`, `500`, `502`, `503`, `504`) are retried up to three times with backoff.
- **Raw-data principle:** the API response is returned without flattening or modifying its weather fields. Transformation is deferred to dbt.

In [3]:
from ingestion.extract import extract_all_cities

rows = extract_all_cities(logical_date)



### Extraction evidence

The following output verifies that one response was returned for every configured city.

In [4]:
len(rows), [row["city"] for row in rows]

(5, ['Amsterdam', 'Berlin', 'Bengaluru', 'New York', 'Sydney'])

The following sample shows the raw daily API payload for one city. At this stage the nested API structure is intentionally preserved.

In [5]:
rows[0]["city"], rows[0]["response"]["daily"]

('Amsterdam',
 {'time': ['2026-09-23'],
  'weather_code': [53],
  'temperature_2m_max': [19.7],
  'temperature_2m_min': [15.1],
  'apparent_temperature_max': [19.4],
  'precipitation_sum': [1.5],
  'rain_sum': [1.5],
  'precipitation_hours': [6.0],
  'sunshine_duration': [14951.52],
  'daylight_duration': [43762.49],
  'wind_speed_10m_max': [12.6],
  'wind_gusts_10m_max': [27.0]})

## 2. Load

The extracted API responses are persisted in `raw.weather_daily`.

- **Target:** `raw.weather_daily`.
- **Grain:** one row per `(city, logical_date)`.
- **Raw payload:** the complete Open-Meteo response is stored as `JSONB`.
- **Metadata:** `city`, `logical_date`, and `loaded_at` are stored separately from the API payload.
- **Idempotency key:** `(city, logical_date)` is the table's primary key.
- **Rerun behaviour:** `INSERT ... ON CONFLICT ... DO UPDATE` refreshes an existing city/date row instead of creating a duplicate.

In [6]:
from ingestion.load import load_weather

loaded_rows = load_weather(rows, logical_date)
loaded_rows

5

### Raw-load evidence

This query shows the rows loaded for the walkthrough logical date.

For five configured cities, the expected result is five rows sharing the same `logical_date`.

In [7]:
pd.set_option("display.max_colwidth", None)

raw_rows = query(
    """
    select
        city,
        logical_date,
        payload,
        loaded_at
    from raw.weather_daily
    where logical_date = %s
    order by city
    """,
    (logical_date,),
)

raw_df = pd.DataFrame(
    raw_rows,
    columns=[
        "City",
        "Logical Date",
        "Payload",
        "Loaded At",
    ],
)

raw_df

,City,Logical Date,Payload,Loaded At
0,Amsterdam,2026-09-23,"{'daily': {'time': ['2026-09-23'], 'rain_sum': [1.5], 'weather_code': [53], 'daylight_duration': [43762.49], 'precipitation_sum': [1.5], 'sunshine_duration': [14951.52], 'temperature_2m_max': [19.7], 'temperature_2m_min': [15.1], 'wind_gusts_10m_max': [27.0], 'wind_speed_10m_max': [12.6], 'precipitation_hours': [6.0], 'apparent_temperature_max': [19.4]}, 'latitude': 52.40773, 'timezone': 'Europe/Amsterdam', 'elevation': 11.0, 'longitude': 4.842301, 'daily_units': {'time': 'iso8601', 'rain_sum': 'mm', 'weather_code': 'wmo code', 'daylight_duration': 's', 'precipitation_sum': 'mm', 'sunshine_duration': 's', 'temperature_2m_max': '°C', 'temperature_2m_min': '°C', 'wind_gusts_10m_max': 'km/h', 'wind_speed_10m_max': 'km/h', 'precipitation_hours': 'h', 'apparent_temperature_max': '°C'}, 'generationtime_ms': 1.0709762573242188, 'utc_offset_seconds': 7200, 'timezone_abbreviation': 'GMT+2'}",2026-09-24 13:42:58.717286+00:00
1,Bengaluru,2026-09-23,"{'daily': {'time': ['2026-09-23'], 'rain_sum': [1.3], 'weather_code': [53], 'daylight_duration': [43600.23], 'precipitation_sum': [1.3], 'sunshine_duration': [31336.18], 'temperature_2m_max': [27.1], 'temperature_2m_min': [20.2], 'wind_gusts_10m_max': [55.4], 'wind_speed_10m_max': [22.4], 'precipitation_hours': [5.0], 'apparent_temperature_max': [28.2]}, 'latitude': 12.970123, 'timezone': 'Asia/Kolkata', 'elevation': 914.0, 'longitude': 77.56364, 'daily_units': {'time': 'iso8601', 'rain_sum': 'mm', 'weather_code': 'wmo code', 'daylight_duration': 's', 'precipitation_sum': 'mm', 'sunshine_duration': 's', 'temperature_2m_max': '°C', 'temperature_2m_min': '°C', 'wind_gusts_10m_max': 'km/h', 'wind_speed_10m_max': 'km/h', 'precipitation_hours': 'h', 'apparent_temperature_max': '°C'}, 'generationtime_ms': 1.3213157653808594, 'utc_offset_seconds': 19800, 'timezone_abbreviation': 'GMT+5:30'}",2026-09-24 13:42:58.717286+00:00
2,Berlin,2026-09-23,"{'daily': {'time': ['2026-09-23'], 'rain_sum': [0.0], 'weather_code': [3], 'daylight_duration': [43764.11], 'precipitation_sum': [0.0], 'sunshine_duration': [39775.27], 'temperature_2m_max': [19.2], 'temperature_2m_min': [7.5], 'wind_gusts_10m_max': [21.2], 'wind_speed_10m_max': [10.3], 'precipitation_hours': [0.0], 'apparent_temperature_max': [17.1]}, 'latitude': 52.54833, 'timezone': 'Europe/Berlin', 'elevation': 38.0, 'longitude': 13.407822, 'daily_units': {'time': 'iso8601', 'rain_sum': 'mm', 'weather_code': 'wmo code', 'daylight_duration': 's', 'precipitation_sum': 'mm', 'sunshine_duration': 's', 'temperature_2m_max': '°C', 'temperature_2m_min': '°C', 'wind_gusts_10m_max': 'km/h', 'wind_speed_10m_max': 'km/h', 'precipitation_hours': 'h', 'apparent_temperature_max': '°C'}, 'generationtime_ms': 1.6843080520629883, 'utc_offset_seconds': 7200, 'timezone_abbreviation': 'GMT+2'}",2026-09-24 13:42:58.717286+00:00
3,New York,2026-09-23,"{'daily': {'time': ['2026-09-23'], 'rain_sum': [0.0], 'weather_code': [3], 'daylight_duration': [43625.81], 'precipitation_sum': [0.0], 'sunshine_duration': [35298.02], 'temperature_2m_max': [19.6], 'temperature_2m_min': [12.1], 'wind_gusts_10m_max': [53.3], 'wind_speed_10m_max': [21.9], 'precipitation_hours': [0.0], 'apparent_temperature_max': [16.5]}, 'latitude': 40.738136, 'timezone': 'America/New_York', 'elevation': 27.0, 'longitude': -74.04254, 'daily_units': {'time': 'iso8601', 'rain_sum': 'mm', 'weather_code': 'wmo code', 'daylight_duration': 's', 'precipitation_sum': 'mm', 'sunshine_duration': 's', 'temperature_2m_max': '°C', 'temperature_2m_min': '°C', 'wind_gusts_10m_max': 'km/h', 'wind_speed_10m_max': 'km/h', 'precipitation_hours': 'h', 'apparent_temperature_max': '°C'}, 'generationtime_ms': 1.4460086822509766, 'utc_offset_seconds': -14400, 'timezone_abbreviation': 'GMT-4'}",2026-09-24 13:42:58.717286+00:00
4,Sydney,2026-09-23,"{'daily': {'time': ['2026-09-23'], 'rain_sum': [4.3], 'weather_code': [61], 'daylight_duration': [43688.0], 'precipitation_sum': [4.3

### Re-run safety

The same extracted data is loaded a second time.

The row count is measured both before and after the second load. Because `(city, logical_date)` is unique and the loader uses an upsert, the expected behaviour is:

- rows before rerun: `5`
- rows after rerun: `5`
- no duplicate city/date records are created

In [8]:
before = query(
    """
    select count(*)
    from raw.weather_daily
    where logical_date = %s
    """,
    (logical_date,),
)[0][0]

load_weather(rows, logical_date)

after = query(
    """
    select count(*)
    from raw.weather_daily
    where logical_date = %s
    """,
    (logical_date,),
)[0][0]

print("Rows before rerun:", before)
print("Rows after rerun: ", after)
print("Idempotent:", before == after)

Rows before rerun: 5
Rows after rerun:  5
Idempotent: True


## 3. Transform and validate with dbt

dbt performs the transformation and data-quality layers after raw ingestion.

### Staging model — `staging.stg_weather`

The staging model:

- reads from the declared `raw.weather_daily` dbt source;
- extracts values from the nested JSON payload;
- applies explicit SQL data types;
- converts sunshine and daylight durations from seconds to hours;
- retains the Airflow `logical_date` and raw `loaded_at` metadata;
- produces one typed weather observation per city and date.

### Mart — `marts.city_weather_profile`

The mart produces one current weather profile per city using the latest **30-calendar-day window**.

It calculates:

- number of observed days in the window;
- maximum-temperature trend in °C per day;
- average absolute day-to-day maximum-temperature change;
- total precipitation;
- longest consecutive rainy-day streak;
- sunshine as a percentage of available daylight.

If a date is missing inside the 30-calendar-day window, the window is not extended backwards. `days_observed` therefore exposes incomplete coverage rather than hiding it.

### Data-quality tests

The dbt test suite checks:

- required fields for null values;
- uniqueness of `(city, weather_date)`;
- consistency between `weather_date` and the ingestion logical date;
- raw-to-staging row-count reconciliation;
- valid latitude and longitude ranges;
- non-negative precipitation and wind values;
- valid precipitation-hour ranges;
- temperature maximum not below temperature minimum;
- sunshine duration not greater than daylight duration;
- mart window and metric validity.

In [9]:
import subprocess

def dbt(*args):
    r = subprocess.run(["dbt", *args], cwd="/opt/airflow/dbt", capture_output=True, text=True)
    print(r.stdout[-4000:])
    if r.returncode != 0:
        print(r.stderr[-2000:])
    return r.returncode


dbt("run")
dbt("test")

13:43:03  Running with dbt=1.8.8
13:43:03  Registered adapter: postgres=1.8.2
13:43:03  Unable to do partial parsing because saved manifest not found. Starting full parse.
13:43:05  [WARNING]: Test 'test.assessment.test_city_weather_profile_valid_values' (tests/marts/test_city_weather_profile_valid_values.sql) depends on a node named 'city_weather_30d_profile' in package '' which was not found
13:43:05  Found 2 models, 26 data tests, 1 source, 424 macros
13:43:05  
13:43:06  Concurrency: 4 threads (target='dev')
13:43:06  
13:43:06  1 of 2 START sql table model staging.stg_weather ............................... [RUN]
13:43:06  1 of 2 OK created sql table model staging.stg_weather .......................... [SELECT 5 in 0.18s]
13:43:06  2 of 2 START sql table model marts.city_weather_profile ........................ [RUN]
13:43:06  2 of 2 OK created sql table model marts.city_weather_profile ................... [SELECT 5 in 0.14s]
13:43:06  
13:43:06  Finished running 2 table models in

0

### Staging evidence

The following query shows that the raw JSON has been converted into typed, analysis-ready columns.

In [10]:
staging_rows = query(
    """
    select
        city,
        weather_date,
        temperature_2m_max,
        temperature_2m_min,
        precipitation_sum,
        sunshine_duration_hours
    from staging.stg_weather
    order by weather_date desc, city
    limit 10
    """
)

staging_df = pd.DataFrame(
    staging_rows,
    columns=[
        "City",
        "Weather Date",
        "Max Temp (°C)",
        "Min Temp (°C)",
        "Precipitation (mm)",
        "Sunshine (hours)",
    ],
)

staging_df

,City,Weather Date,Max Temp (°C),Min Temp (°C),Precipitation (mm),Sunshine (hours)
0,Amsterdam,2026-09-23,19.7,15.1,1.5,4.15
1,Bengaluru,2026-09-23,27.1,20.2,1.3,8.70
2,Berlin,2026-09-23,19.2,7.5,0.0,11.05
3,New York,2026-09-23,19.6,12.1,0.0,9.81
4,Sydney,2026-09-23,19.5,14.3,4.3,7.88


## 4. Orchestration with Airflow

The pipeline is orchestrated by the `weather_daily_pipeline` DAG.

### DAG structure

Each DAG run executes:

`extract → load → dbt run → dbt test`

### Logical-date design

- One DAG run represents exactly one logical date.
- Airflow's `{{ ds }}` value is passed into both extraction and loading.
- Extraction code contains no loop over historical dates.
- Airflow owns historical scheduling and backfill behaviour.

### Daily scheduling

The DAG uses a daily schedule. Normal scheduled execution therefore creates one new logical-date run each day.

### Historical backfill

For the walkthrough, Airflow is asked to backfill the latest 30 complete calendar days.

Airflow creates one DAG run for each logical date in that range. Each run still follows the same four-task dependency chain.

### Concurrency

`max_active_runs=1` intentionally keeps logical-date DAG runs sequential.

The five independent city HTTP requests inside each extract task are parallelized, but dbt executions are not allowed to overlap because every dbt run rebuilds the same staging and mart relations.

### Failure handling

- Airflow tasks have retries.
- Airflow tasks have an execution timeout.
- HTTP requests additionally have request-level retry and timeout handling.
- dbt tests are downstream of dbt model execution, so a failed transformation prevents validation from being reported as successful.

### Execute the 30-day backfill

This step runs the full Airflow DAG for 30 logical dates.

Because each logical date executes `extract → load → dbt run → dbt test` sequentially, this cell can take several minutes to complete.

The subprocess captures Airflow logs while it runs and prints the final portion after completion, including the final backfill status and task counts.

In [11]:
backfill = subprocess.run(
    [
        "airflow",
        "dags",
        "backfill",
        "-s",
        start_date.isoformat(),
        "-e",
        end_date.isoformat(),
        "weather_daily_pipeline",
    ],
    capture_output=True,
    text=True,
)

print(backfill.stdout[-5000:])

if backfill.returncode != 0:
    print(backfill.stderr[-3000:])
    raise RuntimeError("Airflow backfill failed")


iled: 0 | skipped: 0 | deadlocked: 0 | not ready: 1
[2026-09-24T13:51:32.972+0000] {backfill_job_runner.py:453} INFO - [backfill progress] | finished run 29 of 30 | tasks waiting: 1 | succeeded: 118 | running: 1 | failed: 0 | skipped: 0 | deadlocked: 0 | not ready: 1
[2026-09-24T13:51:34.001+0000] {backfill_job_runner.py:453} INFO - [backfill progress] | finished run 29 of 30 | tasks waiting: 1 | succeeded: 118 | running: 1 | failed: 0 | skipped: 0 | deadlocked: 0 | not ready: 1
[2026-09-24T13:51:35.022+0000] {base_executor.py:168} INFO - Adding to queue: ['airflow', 'tasks', 'run', 'weather_daily_pipeline', 'dbt_test', 'backfill__2026-09-23T00:00:00+00:00', '--local', '--pool', 'default_pool', '--subdir', 'DAGS_FOLDER/weather_daily_pipeline.py', '--cfg-path', '/tmp/tmp01u_v2t2']
[2026-09-24T13:51:35.024+0000] {local_executor.py:93} INFO - QueuedLocalWorker running ['airflow', 'tasks', 'run', 'weather_daily_pipeline', 'dbt_test', 'backfill__2026-09-23T00:00:00+00:00', '--local', '--poo

### Evidence after the first backfill

The 30-day backfill should produce:

- 30 distinct logical dates;
- 5 configured cities per date;
- therefore 150 raw rows for a complete window.

The following checks establish the warehouse state after the initial backfill.

In [12]:
before = query(
    """
    select count(*)
    from raw.weather_daily
    where logical_date between %s and %s
    """,
    (start_date, end_date),
)[0][0]

In [13]:
summary_rows = query(
    """
    select
        min(logical_date) as first_date,
        max(logical_date) as last_date,
        count(distinct logical_date) as dates_loaded,
        count(*) as rows_loaded
    from raw.weather_daily
    where logical_date between %s and %s
    """,
    (start_date, end_date),
)

summary_df = pd.DataFrame(
    summary_rows,
    columns=[
        "First Date",
        "Last Date",
        "Dates Loaded",
        "Rows Loaded",
    ],
)

summary_df

,First Date,Last Date,Dates Loaded,Rows Loaded
0,2026-08-25,2026-09-23,30,150


### Completeness by logical date

This query checks how many configured cities were loaded for every date in the backfill range.

For a complete run, every logical date should show `5` city rows.

In [14]:
coverage_rows = query(
    """
    select
        logical_date,
        count(*) as cities_loaded
    from raw.weather_daily
    where logical_date between %s and %s
    group by logical_date
    order by logical_date
    """,
    (start_date, end_date),
)

coverage_df = pd.DataFrame(
    coverage_rows,
    columns=[
        "Logical Date",
        "Cities Loaded",
    ],
)

coverage_df

,Logical Date,Cities Loaded
0,2026-08-25,5
1,2026-08-26,5
2,2026-08-27,5
3,2026-08-28,5
4,2026-08-29,5
5,2026-08-30,5
6,2026-08-31,5
7,2026-09-01,5
8,2026-09-02,5
9,2026-09-03,5


### End-to-end rerun safety

The same 30-day Airflow backfill is executed again.

`--reset-dagruns` resets the Airflow run records so the same logical dates actually execute again. It does **not** delete warehouse rows.

The expected warehouse behaviour is unchanged because raw ingestion uses the `(city, logical_date)` primary key with an upsert.

Therefore:

- first backfill → 150 rows;
- second backfill → still 150 rows;
- existing city/date rows are refreshed rather than duplicated.

In [15]:
rerun = subprocess.run(
    [
        "airflow",
        "dags",
        "backfill",
        "--reset-dagruns",
        "-y",
        "-s",
        start_date.isoformat(),
        "-e",
        end_date.isoformat(),
        "weather_daily_pipeline",
    ],
    capture_output=True,
    text=True,
)

print(rerun.stdout[-5000:])

if rerun.returncode != 0:
    print(rerun.stderr[-3000:])
    raise RuntimeError("Airflow rerun backfill failed")

ocked: 0 | not ready: 1
[2026-09-24T13:59:33.103+0000] {backfill_job_runner.py:453} INFO - [backfill progress] | finished run 29 of 30 | tasks waiting: 1 | succeeded: 118 | running: 1 | failed: 0 | skipped: 0 | deadlocked: 0 | not ready: 1
[2026-09-24T13:59:34.131+0000] {backfill_job_runner.py:453} INFO - [backfill progress] | finished run 29 of 30 | tasks waiting: 1 | succeeded: 118 | running: 1 | failed: 0 | skipped: 0 | deadlocked: 0 | not ready: 1
[2026-09-24T13:59:35.162+0000] {base_executor.py:168} INFO - Adding to queue: ['airflow', 'tasks', 'run', 'weather_daily_pipeline', 'dbt_test', 'backfill__2026-09-23T00:00:00+00:00', '--local', '--pool', 'default_pool', '--subdir', 'DAGS_FOLDER/weather_daily_pipeline.py', '--cfg-path', '/tmp/tmp5mdw9f8g']
[2026-09-24T13:59:35.164+0000] {local_executor.py:93} INFO - QueuedLocalWorker running ['airflow', 'tasks', 'run', 'weather_daily_pipeline', 'dbt_test', 'backfill__2026-09-23T00:00:00+00:00', '--local', '--pool', 'default_pool', '--subdi

In [16]:
after = query(
    """
    select count(*)
    from raw.weather_daily
    where logical_date between %s and %s
    """,
    (start_date, end_date),
)[0][0]

print("Rows before rerun:", before)
print("Rows after rerun: ", after)
print("Idempotent:", before == after)

Rows before rerun: 150
Rows after rerun:  150
Idempotent: True


## 5. Business result

The final mart contains one current rolling weather profile for every configured city.

Each row summarizes up to the latest 30 calendar days and exposes both weather behaviour and data coverage.

The result allows a user to compare cities using:

- temperature direction;
- day-to-day temperature variability;
- total precipitation;
- longest rainy period;
- proportion of daylight that was sunny;
- number and date range of observations used.

In [17]:
mart_rows = query(
    """
    select
        city,
        window_start_date,
        window_end_date,
        days_observed,
        temperature_trend_c_per_day,
        avg_day_to_day_temp_change_c,
        total_precipitation_mm,
        longest_rain_streak_days,
        sunshine_percentage
    from marts.city_weather_profile
    order by city
    """
)

columns = [
    "city",
    "window_start_date",
    "window_end_date",
    "days_observed",
    "temperature_trend_c_per_day",
    "avg_day_to_day_temp_change_c",
    "total_precipitation_mm",
    "longest_rain_streak_days",
    "sunshine_percentage",
]

pd.DataFrame(mart_rows, columns=columns)

,city,window_start_date,window_end_date,days_observed,temperature_trend_c_per_day,avg_day_to_day_temp_change_c,total_precipitation_mm,longest_rain_streak_days,sunshine_percentage
0,Amsterdam,2026-08-25,2026-09-23,30,-0.147,1.72,143.40,13,70.90
1,Bengaluru,2026-08-25,2026-09-23,30,0.025,0.72,100.20,11,85.41
2,Berlin,2026-08-25,2026-09-23,30,-0.234,2.36,47.70,6,78.38
3,New York,2026-08-25,2026-09-23,30,-0.185,2.43,122.90,8,76.16
4,Sydney,2026-08-25,2026-09-23,30,0.115,3.41,39.50,3,86.68


## 6. What I would change at scale

### Ingestion concurrency

For this assessment, each Airflow DAG run processes one logical date. Within the extract task, the API calls for the five cities are made in parallel since they are independent and mostly waiting on network I/O.

I kept the DAG runs themselves sequential because each run also executes dbt against the same staging and mart tables. If the pipeline had to handle a much larger historical load, I would allow the ingestion part to run in parallel and then run the dbt transformation once the required raw data is available.

### Airflow data passing

The amount of data returned for five cities and one date is very small, so passing the extracted responses from the `extract` task to the `load` task through Airflow XCom is reasonable here.

For a larger pipeline, I would avoid putting the actual dataset in XCom. I would write the extracted data to something like object storage and only pass the file path or another reference between Airflow tasks.

### Data quality as the pipeline grows

For this assessment, dbt tests cover the main correctness checks after the data is loaded and transformed.

If this pipeline handled more cities or became a daily production feed, I would add a simple completeness check before transformation, for example verifying that every expected city is present for the logical date before allowing the downstream dbt step to continue.

That would help catch partial API failures early, instead of allowing an incomplete day to flow into the mart and only noticing it later from the final output.